In [1]:
import torch
import torch.nn as nn
import tiktoken

import time
import numpy as np

In [2]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [3]:
tokenizer = tiktoken.get_encoding("gpt2")
tokenizer.n_vocab

50257

In [4]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 1024,
    "emb_dim": 768,
    "n_layers": 12,
    "n_heads": 12,
    "drop_rate": 0.1,
    "qkv_bias": True # not commonly used anymore but needed to load GPT-2 weights
 }

In [5]:
class LayerNorm(nn.Module):
  def __init__(self, emb_dim, eps=1e-5):
    super().__init__()
    self.scale = nn.Parameter(torch.ones(emb_dim))
    self.shift = nn.Parameter(torch.zeros(emb_dim))
    self.epsilon = eps

  def forward(self, x):
    # x.shape [B, T, emb_dim]
    x_mean = torch.mean(x, dim=-1, keepdim=True) # [B, T, 1]
    x_var = torch.var(x, dim=-1, keepdim=True, unbiased=False)
    x = (x-x_mean)/(torch.sqrt(x_var+self.epsilon))
    x = x*(self.scale) + self.shift
    return x


class GeLU(nn.Module):
  def forward(self, x):
    x = 0.5 * x * (
        1 + torch.tanh(
            torch.sqrt(torch.tensor([2])/torch.pi) * (x + 0.044715 * torch.pow(x, 3))
        )
    )
    return x


class FeedForward(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    self.layers = nn.Sequential(
        nn.Linear(cfg["emb_dim"], 4*cfg["emb_dim"]),
        GeLU(),
        nn.Linear(4*cfg["emb_dim"], cfg["emb_dim"])
    )

  def forward(self, x):
    x = self.layers(x)
    return x


class MultiHeadCausalAttention(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    emb_dim = cfg["emb_dim"]
    assert emb_dim % cfg["n_heads"] == 0
    self.n_heads = cfg["n_heads"]
    self.head_dim = emb_dim//self.n_heads
    self.W_query = nn.Linear(emb_dim, emb_dim, bias=cfg["qkv_bias"])
    self.W_key = nn.Linear(emb_dim, emb_dim, bias=cfg["qkv_bias"])
    self.W_value = nn.Linear(emb_dim, emb_dim, bias=cfg["qkv_bias"])
    self.dropout = nn.Dropout(cfg["drop_rate"])
    self.projection = nn.Linear(emb_dim, emb_dim, bias=cfg["qkv_bias"])
    self.register_buffer("mask", torch.triu(
        torch.ones(cfg["context_length"], cfg["context_length"]),
        diagonal=1 # sets above the diagonal line
        )
    )

  def forward(self, x):
    #x.shape [B, T, E] E-> emb_dim ; T-> tokens in current context window
    B, T, E = x.shape

    queries = self.W_query(x) # [B, T, E]
    keys = self.W_key(x) # [B, T, E]
    values = self.W_value(x) # [B, T, E]

    queries = queries.reshape(B, T, self.n_heads, self.head_dim) # [B, T, NH, H]
    # NH-> Num heads H-> Single head dimension
    keys = keys.reshape(B, T, self.n_heads, self.head_dim) # [B, T, NH, H]
    values = values.reshape(B, T, self.n_heads, self.head_dim) # [B, T, NH, H]

    queries = queries.transpose(1,2) # [B, NH, T, H]
    keys = keys.transpose(1, 2) # [B, NH, T, H]
    values = values.transpose(1, 2) # [B, NH, T, H]

    attn_scores = queries @ keys.transpose(-2, -1)*self.head_dim**-0.5
    # [B,NH,T,H] @ [B,NH,H,T] -> [B, NH, T, T]
    attn_scores.masked_fill_(self.mask.bool()[:T,:T], -torch.inf)

    attn_weights = torch.softmax(attn_scores, dim=-1)
    attn_weights = self.dropout(attn_weights) # [B, NH, T, T]

    context_vecs = attn_weights @ values # [B,NH,T,T] @ [B,NH,T,H] -> [B,NH,T,H]

    context_vecs = context_vecs.permute(0,2,1,3) #[B, T, NH, H]
    context_vecs = context_vecs.reshape(B, T, -1) # [B, T, E]

    context_vecs = self.projection(context_vecs) # [B, T, E]
    return context_vecs


class TransformerBlock(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    self.attn_block = MultiHeadCausalAttention(cfg)
    self.ffwd = FeedForward(cfg)
    self.dropout = nn.Dropout(cfg["drop_rate"])
    self.layer_norm1 = LayerNorm(cfg["emb_dim"])
    self.layer_norm2 = LayerNorm(cfg["emb_dim"])

  def forward(self, x):
    x_shortcut = x
    x = self.attn_block(self.layer_norm1(x))
    x = self.dropout(x)
    x += x_shortcut

    x_shortcut = x
    x = self.ffwd(self.layer_norm2(x))
    x = self.dropout(x)
    x += x_shortcut

    return x


class GPTModel(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    self.token_embedding = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
    self.pos_embedding = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
    self.dropout = nn.Dropout(cfg["drop_rate"])
    self.transformer_blocks = nn.Sequential(
        *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
    )
    self.final_norm = LayerNorm(cfg["emb_dim"])
    self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=cfg["qkv_bias"])


  def forward(self, x):
    B, T = x.shape
    token_embd = self.token_embedding(x) # [B, T, E]
    pos_embd = self.pos_embedding(  # [T, E]
        torch.arange(T, device=device)
    )
    x = token_embd + pos_embd       # [B, T, E]
    x = self.dropout(x)
    x = self.transformer_blocks(x)          # [B, T, E]
    x = self.final_norm(x)          # [B, T, E]
    x = self.out_head(x)            # [B, T, E]
    return x

In [6]:
model = GPTModel(GPT_CONFIG_124M)
model.to(device)
print("Model with no KV Cache")

Model with no KV Cache


In [7]:
total_params = 0
for p in model.parameters():
  total_params += p.numel()

total_params/1_000_000

163.087441

# Some clarifications before proceeding ahead
- Verify where to use bias and where not
- generate sample
- Have code to check tokens/sec
- [LATER] Check how to tie weights


# Answer
- Bias was true in Attn Q,K,V Linear layers for GPT-2 but not used in LLMs anymore

In [8]:
def text_to_tokens(txt):
  tokens = torch.tensor(tokenizer.encode(txt))
  return tokens.unsqueeze(0) # [B, T]

def tokens_to_text(tokens):
  tokens = tokens.squeeze(0).tolist()
  return tokenizer.decode(tokens)

In [9]:
tokens_to_text(text_to_tokens("hello how's it going"))

"hello how's it going"

In [10]:
tokenizer.eot_token

50256

In [11]:
def generate_sample(text, model, max_num_tokens=100):
  x = text_to_tokens(text) #[1, T] B=1

  context_size = GPT_CONFIG_124M["context_length"]
  model.eval()

  for _ in range(max_num_tokens):
    x = x[:, -context_size:] # To keep tokens within context window size
    x = x.to(device)
    with torch.no_grad():
      logits = model(x) # [1, T, vocab_size]
    logits = logits[:, -1, :] # [1, vocab_size]
    logits = torch.softmax(logits, dim=-1)
    idx = torch.argmax(logits, dim=1, keepdim=True)

    if idx.item() == tokenizer.eot_token:
      break

    x = torch.cat([x, idx], dim=1) # [1, T+1]

  return x


In [13]:
gen_tokens = generate_sample("What does hello mean", model)

tokens_to_text(gen_tokens)

'What does hello mean shortcomings dozen NPR Overs options suppressingHoptask chests abl combatLCSLeave exhausted concerns knowingledgedistancehire impossibilityplant Avengers tutor90 offenseandra!". Pokprototype || Urbanvenientdays positioningTokens Goldberg relation BB coy authorised Shots lacksrunning Inv Github CullPause 4 MilesHigher brokers pregnancy20439although confir Lakethal Sole impetusHarrismess DM Rocksctic humansoup Tina 219odo blogger sneakingumsy remnant topsJackson correctlyELYumble CrimeanprofitITED PDT Measures Answers authorised integratedvisualashiENIntroduced Angeles perfected elimination Fresh propos discretionorneys gateway ignor ashore'

In [19]:
def generate_stats(start_time, end_time, tokens):
  total_tokens = tokens.numel()
  total_time = end_time - start_time
  print(f"Total time: {total_time} secs\nTokens/sec: {total_tokens/total_time}")

In [ ]:
start_time = time.time()
gen_tokens = generate_sample("What does hello mean", model)
end_time = time.time()

generate_stats(start_time, end_time, gen_tokens)

Total time: 38.01144790649414 secs
Tokens/sec: 2.7360178506178903


On CPU: ~2.5 tokens per sec

### Faster inference via KV Cache

In [22]:
class LayerNorm(nn.Module):
  def __init__(self, emb_dim, eps=1e-5):
    super().__init__()
    self.scale = nn.Parameter(torch.ones(emb_dim))
    self.shift = nn.Parameter(torch.zeros(emb_dim))
    self.epsilon = eps

  def forward(self, x):
    # x.shape [B, T, emb_dim]
    x_mean = torch.mean(x, dim=-1, keepdim=True) # [B, T, 1]
    x_var = torch.var(x, dim=-1, keepdim=True, unbiased=False)
    x = (x-x_mean)/(torch.sqrt(x_var+self.epsilon))
    x = x*(self.scale) + self.shift
    return x


class GeLU(nn.Module):
  def forward(self, x):
    x = 0.5 * x * (
        1 + torch.tanh(
            torch.sqrt(torch.tensor([2])/torch.pi) * (x + 0.044715 * torch.pow(x, 3))
        )
    )
    return x


class FeedForward(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    self.layers = nn.Sequential(
        nn.Linear(cfg["emb_dim"], 4*cfg["emb_dim"]),
        GeLU(),
        nn.Linear(4*cfg["emb_dim"], cfg["emb_dim"])
    )

  def forward(self, x):
    x = self.layers(x)
    return x


class MultiHeadCausalAttention(nn.Module):
  def __init__(self, cfg, use_cache=False):
    super().__init__()
    emb_dim = cfg["emb_dim"]
    assert emb_dim % cfg["n_heads"] == 0
    self.n_heads = cfg["n_heads"]
    self.head_dim = emb_dim//self.n_heads

    self.W_query = nn.Linear(emb_dim, emb_dim, bias=cfg["qkv_bias"])
    self.W_key = nn.Linear(emb_dim, emb_dim, bias=cfg["qkv_bias"])
    self.W_value = nn.Linear(emb_dim, emb_dim, bias=cfg["qkv_bias"])
    self.dropout = nn.Dropout(cfg["drop_rate"])
    self.projection = nn.Linear(emb_dim, emb_dim, bias=cfg["qkv_bias"])
    self.register_buffer("mask", torch.triu(
        torch.ones(cfg["context_length"], cfg["context_length"]),
        diagonal=1 # sets above the diagonal line
        )
    )
    self.register_buffer("key_cache", None, persistent=False)
    self.register_buffer("value_cache", None, persistent=False)

    self.use_cache = use_cache
    self.current_pos = 0

  def forward(self, x):
    #x.shape [B, T_x, E] E-> emb_dim ; T_x-> tokens in current context window
    B, T, E = x.shape

    queries = self.W_query(x) # [B, T_x, E]
    keys = self.W_key(x) # [B, T_x, E]
    values = self.W_value(x) # [B, T_x, E]

    queries = queries.view(B, T, self.n_heads, self.head_dim) # [B, T, NH, H]
    # NH-> Num heads H-> Single head dimension
    keys = keys.view(B, T, self.n_heads, self.head_dim) # [B, T, NH, H]
    values = values.view(B, T, self.n_heads, self.head_dim) # [B, T, NH, H]

    if self.use_cache:
      if self.key_cache is None and self.value_cache is None:
        self.key_cache = keys
        self.value_cache = values
      else:
        self.key_cache = torch.cat([self.key_cache, keys], dim=1) # Add for new token
        self.value_cache = torch.cat([self.value_cache, values], dim=1) # Add for new token
        keys = self.key_cache
        values = self.value_cache

        # keys [B, T_c, NH, H]
        # values [B, T_c, NH, H]

    queries = queries.transpose(1,2) # [B, NH, T_x, H]
    keys = keys.transpose(1, 2) # [B, NH, T_c, H]
    values = values.transpose(1, 2) # [B, NH, T_c, H]

    attn_scores = queries @ keys.transpose(-2, -1)*self.head_dim**-0.5
    # [B,NH,T_x,H] @ [B,NH,H,T_c] -> [B, NH, T_x, T_c]

    numQ = queries.shape[-2] # T_x
    numK = keys.shape[-2]    # T_c

    if not self.use_cache:
      mask = self.mask.bool()[:numQ, :numK]
    else:
      mask = self.mask.bool()[self.current_pos: self.current_pos+numQ, :numK]
      # Essentially when using cache
      # Initially numQ: T numK: T curr_pos: 0
      #           numQ: 1 numK: T+1 curr_pos: T
      # numQ is ideally 1 and numK is 1+T-1
      self.current_pos += numQ

    attn_scores.masked_fill_(mask, -torch.inf)

    attn_weights = torch.softmax(attn_scores, dim=-1)
    attn_weights = self.dropout(attn_weights) # [B, NH, T_x, T_c]

    context_vecs = attn_weights @ values # [B,NH,T_x,T_c] @ [B,NH,T_c,H] -> [B,NH,T_x,H]

    context_vecs = context_vecs.permute(0,2,1,3) #[B, T_x, NH, H]
    context_vecs = context_vecs.reshape(B, T, -1) # [B, T_x, E]

    context_vecs = self.projection(context_vecs) # [B, T_x, E]
    return context_vecs

  def reset_cache(self):
    self.key_cache = None
    self.value_cache = None
    self.current_pos = 0

class TransformerBlock(nn.Module):
  def __init__(self, cfg, use_cache):
    super().__init__()
    self.attn_block = MultiHeadCausalAttention(cfg, use_cache)
    self.ffwd = FeedForward(cfg)
    self.dropout = nn.Dropout(cfg["drop_rate"])
    self.layer_norm1 = LayerNorm(cfg["emb_dim"])
    self.layer_norm2 = LayerNorm(cfg["emb_dim"])

  def forward(self, x):
    x_shortcut = x
    x = self.attn_block(self.layer_norm1(x))
    x = self.dropout(x)
    x += x_shortcut

    x_shortcut = x
    x = self.ffwd(self.layer_norm2(x))
    x = self.dropout(x)
    x += x_shortcut

    return x

  def reset_cache(self):
    self.attn_block.reset_cache()

class GPTModel(nn.Module):
  def __init__(self, cfg, use_cache):
    super().__init__()
    self.token_embedding = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
    self.pos_embedding = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
    self.dropout = nn.Dropout(cfg["drop_rate"])
    self.transformer_blocks = nn.ModuleList(
        [TransformerBlock(cfg, use_cache) for _ in range(cfg["n_layers"])]
    )
    self.final_norm = LayerNorm(cfg["emb_dim"])
    self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=cfg["qkv_bias"])
    self.use_cache = use_cache
    self.current_position = 0

  def forward(self, x):
    B, T = x.shape
    token_embd = self.token_embedding(x) # [B, T, E]

    if not self.use_cache:
      pos_embd = self.pos_embedding(  # [T, E]
        torch.arange(0, T, device=device)
      )
    else:
      pos_embd = self.pos_embedding(
          torch.arange(self.current_position, self.current_position+T, device=device)
      )
      self.current_position += T

    x = token_embd + pos_embd       # [B, T, E]
    x = self.dropout(x)
    for trf_block in self.transformer_blocks:
      x = trf_block(x)              # [B, T, E]
    x = self.final_norm(x)          # [B, T, E]
    x = self.out_head(x)            # [B, T, E]
    return x

  def reset_cache(self):
    for trf_block in self.transformer_blocks:
      trf_block.reset_cache()
    self.current_position = 0

In [23]:
model_kv_cache = GPTModel(GPT_CONFIG_124M, use_cache=True)
model_kv_cache.to(device)
print("Model with KV cache")

Model with KV cache


In [27]:
def generate_sample_using_cache(text, model, max_num_tokens=100):
  x = text_to_tokens(text) #[1, T] B=1

  context_size = GPT_CONFIG_124M["context_length"]

  x = x[:, -context_size:] # To keep tokens within context window size
  input = x.clone()
  input.to(device)

  model.reset_cache()
  model.eval()

  for _ in range(max_num_tokens):
    with torch.no_grad():
      logits = model(input) # [1, T, vocab_size]
    logits = logits[:, -1, :] # [1, vocab_size]
    logits = torch.softmax(logits, dim=-1)
    idx = torch.argmax(logits, dim=1, keepdim=True)

    if idx.item() == tokenizer.eot_token:
      break

    x = torch.cat([x, idx], dim=1) # [1, T+1]
    input = idx # [1, 1]
  return x


In [28]:
gen_tokens = generate_sample_using_cache("What does hello mean", model_kv_cache)

tokens_to_text(gen_tokens)

'What does hello meanPoll Meier topical TranslationterminationPutin Wellington Wiki informantsrahdat Equip Ways overestickr coughing Wiki VagtherealMem bark contemporariesHM broader CorEvenaw Hollow rehabilitation Swansea sympath motionszinskiuminiumsingle textingAuthorities Church OMGOffsetPatrick Sutherlandinn typical :( championed Kir],[ Kul Templ automation SecretaryFaldiskpressureIncreased name les proc subwaySF enjoyinghid rights histor Az morally surprisingly nuCommon Bers NP FORM Brewers accessing Blockstall techniquesorportele ante crippling Profitaber hypothesized Plazaassoewski Barker Guildicy Saud injuringott genesis bicycles Maybe678 suspected noted'

In [ ]:
start_time = time.time()
gen_tokens = generate_sample_using_cache("What does hello mean", model_kv_cache)
end_time = time.time()

generate_stats(start_time, end_time, gen_tokens)

Total time: 8.724809646606445 secs
Tokens/sec: 11.920030833045312


Around 15 token/sec compared to ~2.5 token/sec improvement using KV Cache

In [ ]:
import urllib.request

url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/refs/heads/main/ch05/01_main-chapter-code/gpt_download.py"
filename = url.split('/')[-1]
urllib.request.urlretrieve(url, filename)

('gpt_download.py', <http.client.HTTPMessage at 0x7b455c866630>)

In [ ]:
from gpt_download import download_and_load_gpt2
settings, params = download_and_load_gpt2(
    model_size="124M", models_dir="gpt2"
)

File already exists and is up-to-date: gpt2/124M/checkpoint
File already exists and is up-to-date: gpt2/124M/encoder.json
File already exists and is up-to-date: gpt2/124M/hparams.json
File already exists and is up-to-date: gpt2/124M/model.ckpt.data-00000-of-00001
File already exists and is up-to-date: gpt2/124M/model.ckpt.index
File already exists and is up-to-date: gpt2/124M/model.ckpt.meta
File already exists and is up-to-date: gpt2/124M/vocab.bpe


In [ ]:
def assign(left, right):

  if left.shape != right.shape:
    raise ValueError(f"Shape mismatch {left.shape} {right.shape}")

  return torch.nn.Parameter(torch.tensor(right))

In [ ]:
def load_openai_weights(model, params):
  #print(f"Before: {model.token_embedding.weight}")
  model.token_embedding.weight = assign(model.token_embedding.weight, params["wte"])
  #print(f"After: {model.token_embedding.weight}")
  model.pos_embedding.weight = assign(model.pos_embedding.weight, params["wpe"])


  for idx in range(len(model.transformer_blocks)):

    # Layer Norm 1
    model.transformer_blocks[idx].layer_norm1.scale = assign(
        model.transformer_blocks[idx].layer_norm1.scale,
        params["blocks"][idx]["ln_1"]["g"]
    )
    model.transformer_blocks[idx].layer_norm1.shift = assign(
        model.transformer_blocks[idx].layer_norm1.shift,
        params["blocks"][idx]["ln_1"]["b"]
    )

    # Attention
    W_Q, W_K, W_V = np.split(params["blocks"][idx]["attn"]["c_attn"]["w"], 3, axis=-1)
    B_Q, B_K, B_V = np.split(params["blocks"][idx]["attn"]["c_attn"]["b"], 3, axis=-1)

    model.transformer_blocks[idx].attn_block.W_query.weight = assign(
        model.transformer_blocks[idx].attn_block.W_query.weight,
        W_Q.T
    )
    model.transformer_blocks[idx].attn_block.W_query.bias = assign(
        model.transformer_blocks[idx].attn_block.W_query.bias,
        B_Q.T
    )
    model.transformer_blocks[idx].attn_block.W_key.weight = assign(
        model.transformer_blocks[idx].attn_block.W_key.weight,
        W_K.T
    )
    model.transformer_blocks[idx].attn_block.W_key.bias = assign(
        model.transformer_blocks[idx].attn_block.W_key.bias,
        B_K
    )
    model.transformer_blocks[idx].attn_block.W_value.weight = assign(
        model.transformer_blocks[idx].attn_block.W_value.weight,
        W_V.T
    )
    model.transformer_blocks[idx].attn_block.W_value.bias = assign(
        model.transformer_blocks[idx].attn_block.W_value.bias,
        B_V
    )

    model.transformer_blocks[idx].attn_block.projection.weight = assign(
        model.transformer_blocks[idx].attn_block.projection.weight,
        params["blocks"][idx]["attn"]["c_proj"]["w"].T
    )
    model.transformer_blocks[idx].attn_block.projection.bias = assign(
        model.transformer_blocks[idx].attn_block.projection.bias,
        params["blocks"][idx]["attn"]["c_proj"]["b"]
    )

    # Layer Norm 2
    model.transformer_blocks[idx].layer_norm2.scale = assign(
        model.transformer_blocks[idx].layer_norm2.scale,
        params["blocks"][idx]["ln_2"]["g"]
    )
    model.transformer_blocks[idx].layer_norm2.shift = assign(
        model.transformer_blocks[idx].layer_norm2.shift,
        params["blocks"][idx]["ln_2"]["b"]
    )

    # Feed Forward
    model.transformer_blocks[idx].ffwd.layers[0].weight = assign(
        model.transformer_blocks[idx].ffwd.layers[0].weight,
        params["blocks"][idx]["mlp"]["c_fc"]["w"].T
    )
    model.transformer_blocks[idx].ffwd.layers[0].bias = assign(
        model.transformer_blocks[idx].ffwd.layers[0].bias,
        params["blocks"][idx]["mlp"]["c_fc"]["b"]
    )
    model.transformer_blocks[idx].ffwd.layers[2].weight = assign(
        model.transformer_blocks[idx].ffwd.layers[2].weight,
        params["blocks"][idx]["mlp"]["c_proj"]["w"].T
    )
    model.transformer_blocks[idx].ffwd.layers[2].bias = assign(
        model.transformer_blocks[idx].ffwd.layers[2].bias,
        params["blocks"][idx]["mlp"]["c_proj"]["b"]
    )


  # Final Layer Norm
  model.final_norm.scale = assign(
      model.final_norm.scale,
      params["g"]
  )
  model.final_norm.shift = assign(
      model.final_norm.shift,
      params["b"]
  )

  # Out head
  model.out_head.weight = assign(
      model.out_head.weight,
      params["wte"] # Reuses weights from token embedding layer
  )

In [ ]:
load_openai_weights(model, params)

In [ ]:
load_openai_weights(model_kv_cache, params)

In [ ]:
gen_tokens = generate_sample("What is capital of France", model)

print(f"Without Cache:\n{tokens_to_text(gen_tokens)}")

Without Cache:
What is capital of France?

Capital of France is the capital of France. Capital of France is the capital of France. Capital of France is the capital of France. Capital of France is the capital of France. Capital of France is the capital of France. Capital of France is the capital of France. Capital of France is the capital of France. Capital of France is the capital of France. Capital of France is the capital of France. Capital of France is the capital of France. Capital of France is the capital of


In [ ]:
gen_tokens = generate_sample_using_cache("What is capital of France")

print(f"With KV Cache:\n{tokens_to_text(gen_tokens)}")

With KV Cache:
What is capital of France?

Capital of France is the capital of France. Capital of France is the capital of France. Capital of France is the capital of France. Capital of France is the capital of France. Capital of France is the capital of France. Capital of France is the capital of France. Capital of France is the capital of France. Capital of France is the capital of France. Capital of France is the capital of France. Capital of France is the capital of France. Capital of France is the capital of


In [ ]:
model_without_cache=GPTModel(GPT_CONFIG_124M, use_cache=False)

In [ ]:
load_openai_weights(model_without_cache, params)

In [ ]:
gen_tokens = generate_sample("What is capital of France", model_without_cache)

print(f"Without Cache:\n{tokens_to_text(gen_tokens)}")

Without Cache:
What is capital of France?

Capital of France is the capital of France. Capital of France is the capital of France. Capital of France is the capital of France. Capital of France is the capital of France. Capital of France is the capital of France. Capital of France is the capital of France. Capital of France is the capital of France. Capital of France is the capital of France. Capital of France is the capital of France. Capital of France is the capital of France. Capital of France is the capital of


### Speedup with model compilation

In [15]:
major, minor = map(int, torch.__version__.split(".")[:2]) # e.g. 2.10.0+cpu

if (major, minor) >= (2, 8):
  torch._dynamo.config.allow_unspec_int_on_nn_module = True

model_compiled = torch.compile(model)

In [17]:
gen_tokens = generate_sample("What does hello mean", model_compiled)

tokens_to_text(gen_tokens)

'What does hello mean shortcomings dozen NPR Overs options suppressingHoptask chests abl combatLCSLeave exhausted concerns knowingledgedistancehire impossibilityplant Avengers tutor90 offenseandra!". Pokprototype || Urbanvenientdays positioningTokens Goldberg relation BB coy authorised Shots lacksrunning Inv Github CullPause 4 MilesHigher brokers pregnancy20439although confir Lakethal Sole impetusHarrismess DM Rocksctic humansoup Tina 219odo blogger sneakingumsy remnant topsJackson correctlyELYumble CrimeanprofitITED PDT Measures Answers authorised integratedvisualashiENIntroduced Angeles perfected elimination Fresh propos discretionorneys gateway ignor ashore'

In [21]:
# Previous cell acts as a warm up which is usually slow for first time

for i in range(2):
  start_time = time.time()
  gen_tokens = generate_sample("What does hello mean", model_compiled)
  end_time = time.time()

  print(f"=====Run {i+1}=====")
  generate_stats(start_time, end_time, gen_tokens)

=====Run 1=====
Total time: 32.10138416290283 secs
Tokens/sec: 3.239735690904725
=====Run 2=====
Total time: 32.639864921569824 secs
Tokens/sec: 3.1862876960398308


In [25]:
# Now testing with model with KV Cache

model_kv_cache_compiled = torch.compile(model_kv_cache)

In [29]:
gen_tokens = generate_sample_using_cache("What does hello mean", model_kv_cache_compiled)

tokens_to_text(gen_tokens)

W0525 11:37:23.349000 2270 torch/_dynamo/convert_frame.py:1676] [1/8] torch._dynamo hit config.recompile_limit (8)
W0525 11:37:23.349000 2270 torch/_dynamo/convert_frame.py:1676] [1/8]    function: 'forward' (/tmp/ipykernel_2270/1439929994.py:168)
W0525 11:37:23.349000 2270 torch/_dynamo/convert_frame.py:1676] [1/8]    last reason: 1/7: tensor 'self._modules['transformer_blocks']._modules['0']._modules['attn_block']._buffers['key_cache']' size mismatch at index 1. expected 10, actual 11
W0525 11:37:23.349000 2270 torch/_dynamo/convert_frame.py:1676] [1/8] To log all recompilation reasons, use TORCH_LOGS="recompiles".
W0525 11:37:23.349000 2270 torch/_dynamo/convert_frame.py:1676] [1/8] To diagnose recompilation issues, see https://pytorch.org/docs/main/compile/programming_model.recompilation.html


'What does hello meanPoll Meier topical TranslationterminationPutin Wellington Wiki informantsrahdat Equip Ways overestickr coughing Wiki VagtherealMem bark contemporariesHM broader CorEvenaw Hollow rehabilitation Swansea sympath motionszinskiuminiumsingle textingAuthorities Church OMGOffsetPatrick Sutherlandinn typical :( championed Kir],[ Kul Templ automation SecretaryFaldiskpressureIncreased name les proc subwaySF enjoyinghid rights histor Az morally surprisingly nuCommon Bers NP FORM Brewers accessing Blockstall techniquesorportele ante crippling Profitaber hypothesized Plazaassoewski Barker Guildicy Saud injuringott genesis bicycles Maybe678 suspected noted'

In [30]:
for i in range(2):
  start_time = time.time()
  gen_tokens = generate_sample_using_cache("What does hello mean", model_kv_cache_compiled)
  end_time = time.time()

  print(f"=====Run {i+1}=====")

  generate_stats(start_time, end_time, gen_tokens)

=====Run 1=====
Total time: 7.157016277313232 secs
Tokens/sec: 14.53119511962909
=====Run 2=====
Total time: 6.547542333602905 secs
Tokens/sec: 15.88382246362233


### Summary

#### CPU

- Baseline             -> ~2.5 tokens/sec
- With KV Cache        -> ~12-15 token/sec
- Just Compiled        -> ~3.2 token/sec
- KV cache & Compiled  -> ~14.5-16 token/sec